# Noncompliance Analysis

In this experiment, we want to evaluate the impact of **compliance** and understand how non-compliance can affect or bias the estimated outcome, even when a randomized experiment or randomized controlled trial (RCT) is conducted.

## Experimental Design

We assume that we are working as a DoorDash data scientist evaluating a new payment feature.

DoorDash introduces a feature that allows users to **link their Venmo account and use Venmo to pay for food orders**.

The main research question is:

> **Does actually linking Venmo cause users to generate more revenue for DoorDash?**

To investigate this question, DoorDash runs a randomized experiment with **10,000 users**.

Half of the users are randomly assigned to the treatment group:

$$
T = 1
$$

These users are given the option to link their Venmo account.

The other half are assigned to the control group:

$$
T = 0
$$

These users are not given the option to link Venmo.

Therefore, the randomized assignment is:

$$
5{,}000 \text{ treatment users}, \qquad 5{,}000 \text{ control users}
$$

## The Non-Compliance Problem

The main challenge in this experiment is **non-compliance**.

Even if a user is assigned to the treatment group, DoorDash cannot force that user to actually link their Venmo account.

For example, suppose:

* 5,000 users are assigned to the treatment group.
* Only 214 of those users actually link their Venmo account.
* The remaining 4,786 treatment-assigned users do not link Venmo.

Therefore, the treatment-group compliance rate is:

$$
\text{Compliance Rate}
=
\frac{214}{5000}
\approx 0.0428
=
4.28\%
$$

This means that only about **4.28% of treatment-assigned users actually comply with the treatment**.

This is a very low compliance rate.

## Variables in the Dataset

There are five variables in the dataset:

* **`revenue`**: The amount of revenue generated by the user for DoorDash.
* **`treatment`**: Indicates whether the user was assigned to the Venmo treatment.
* **`compliance`**: Indicates whether the user actually linked their Venmo account.
* **`age`**: The user's age.
* **`signal`**: A variable that will be used later; for now, we ignore it.

## Main Challenge

The important distinction in this experiment is between **treatment assignment** and **actual treatment received**.

Treatment assignment is randomized:

$$
5{,}000 \text{ treatment}, \qquad 5{,}000 \text{ control}
$$

Therefore, the variable `treatment` is randomly assigned and is not systematically related to user characteristics.

However, **actual Venmo usage is not randomized**.

Users in the treatment group decide for themselves whether they want to link their Venmo account.

Therefore:

$$
\text{Randomized Treatment Assignment}
\neq
\text{Randomized Compliance}
$$

Only users assigned to the treatment group have the opportunity to link Venmo, and among those users, only a small proportion actually do so.

This creates the **non-compliance problem**.

As a result, simply comparing users who linked Venmo with users who did not link Venmo may produce a biased estimate of the causal effect because the decision to comply may be related to other user characteristics that also affect revenue.


## 1. As-Treated and Per-Protocol Estimators

We calculate two different estimators to examine the effect of actually linking Venmo: the **As-Treated estimator** and the **Per-Protocol estimator**.

### 1. As-Treated Estimator

For the **As-Treated estimator**, we ignore what users were originally randomly assigned to and compare them based on what they **actually did**.

We compare users who actually linked Venmo:

$$
C = 1
$$

with users who did not link Venmo:

$$
C = 0
$$

Therefore, the As-Treated estimator is:

$$
\text{As-Treated Effect}
=
E[Y \mid C=1]
-
E[Y \mid C=0]
$$

where:

* $Y$ represents user revenue.
* $C$ represents compliance, or whether the user actually linked Venmo.

In simple terms, we compare the **average revenue of users who actually linked Venmo** with the **average revenue of users who did not link Venmo**.

The original randomized treatment assignment is ignored in this comparison.

---

### 2. Per-Protocol Estimator

For the **Per-Protocol estimator**, we only keep users who **followed the protocol associated with their original treatment assignment**.

#### Treatment Group

For users assigned to treatment, following the protocol means actually linking Venmo.

Therefore, we keep users with:

$$
T=1,\qquad C=1
$$

In our experiment, this corresponds to the **214 treatment users who actually linked Venmo**.

#### Control Group

Users assigned to the control group were not given the option to link Venmo.

Therefore, following the control protocol means:

$$
T=0,\qquad C=0
$$

All 5,000 control users satisfy this condition, so they are kept in the analysis.

The Per-Protocol estimator therefore compares:

$$
\text{Treatment Compliers}
\quad \text{vs.} \quad
\text{Control Users}
$$

Mathematically:

$$
\text{Per-Protocol Effect}
=
E[Y \mid T=1,C=1]
-
E[Y \mid T=0,C=0]
$$

In simple terms, we compare the **average revenue of treatment users who actually linked Venmo** with the **average revenue of control users who followed their assigned protocol**.

---

## Key Difference Between the Two Estimators

The **As-Treated estimator** compares users based only on their actual behavior:

$$
C=1
\quad \text{vs.} \quad
C=0
$$

regardless of their original treatment assignment.

The **Per-Protocol estimator** compares only users who followed their assigned treatment protocol:

$$
(T=1,C=1)
\quad \text{vs.} \quad
(T=0,C=0)
$$

Therefore:

* **As-Treated:** Group users according to what they actually did.
* **Per-Protocol:** Keep only users who followed their assigned protocol.

An important issue with both estimators is that **compliance is not randomly assigned**. Users who choose to link Venmo may be systematically different from users who do not. Therefore, both estimators can suffer from **selection bias** and may not represent the true causal effect of linking Venmo.


In [4]:
import numpy as np 
import pandas as pd

df = pd.read_csv("Data_Compliance.csv")


mean_revenue_linked = df[df["compliance"] == 1]["revenue"].mean()
mean_revenue_not_linked = df[df["compliance"] == 0]["revenue"].mean()

as_treated = mean_revenue_linked - mean_revenue_not_linked

print("Mean revenue - Linked Venmo:", mean_revenue_linked)
print("Mean revenue - Not linked:", mean_revenue_not_linked)
print("As-treated effect:", as_treated)


# Per-protocol estimator

# Treatment users who actually complied
treatment_compliers = df[
    (df["treatment"] == 1) &
    (df["compliance"] == 1)
]["revenue"].mean()

# Control users who followed control assignment
control_compliers = df[
    (df["treatment"] == 0) &
    (df["compliance"] == 0)
]["revenue"].mean()

per_protocol = treatment_compliers - control_compliers

print("\nTreatment compliers revenue:", treatment_compliers)
print("Control group revenue:", control_compliers)
print("Per-protocol effect:", per_protocol)


age_linked = df[df["compliance"] == 1]["age"].mean()
age_not_linked = df[df["compliance"] == 0]["age"].mean()

print("\nMean age - Linked Venmo:", age_linked)
print("Mean age - Not linked:", age_not_linked)

Mean revenue - Linked Venmo: 24.716349613454373
Mean revenue - Not linked: 17.555007068008255
As-treated effect: 7.161342545446118

Treatment compliers revenue: 24.716349613454373
Control group revenue: 17.58380804940653
Per-protocol effect: 7.132541564047841

Mean age - Linked Venmo: 30.411214953271028
Mean age - Not linked: 41.496730022481096


## Why As-Treated and Per-Protocol Can Be Biased

Both the **As-Treated** and **Per-Protocol** approaches have essentially the same major problem:

They treat actual compliance as if it were random.

However, compliance is **not random**.

Users decide for themselves whether they want to link Venmo.

For example, younger users may be more familiar or more comfortable with Venmo and therefore may be more likely to link their account.

Our dataset provides evidence of this.

The average age of users who linked Venmo is:

$$
30.4
$$

The average age of users who did not link Venmo is:

$$
41.5
$$

This is a difference of approximately:

$$
41.5 - 30.4 = 11.1 \text{ years}
$$

This is a very large difference.

Therefore, users who comply with the treatment are clearly different from users who do not comply.

In this case, age appears to be related to compliance:

$$
\text{Age} \rightarrow \text{Compliance}
$$

At the same time, age may also affect purchasing behavior and revenue:

$$
\text{Age} \rightarrow \text{Revenue}
$$

This creates a potential confounding relationship:

$$
\text{Age} \rightarrow \text{Compliance}
$$

$$
\text{Age} \rightarrow \text{Revenue}
$$

Therefore, if users who linked Venmo have higher revenue, we cannot immediately conclude that Venmo caused the increase.

Part of the observed revenue difference may instead come from the fact that **different types of users choose to link Venmo**.

For example, if younger users are both more likely to use Venmo and more likely to spend more on DoorDash, then the observed revenue difference may partly reflect the effect of age rather than the effect of Venmo.

This is called **selection bias**.

In simple terms:

> Users who choose to link Venmo are systematically different from users who do not link Venmo.

Therefore, simply comparing compliers and non-compliers can produce a biased estimate of the causal effect of Venmo.


In [5]:
treatment_mean = df[df["treatment"] == 1]["revenue"].mean()
control_mean = df[df["treatment"] == 0]["revenue"].mean()

ITT = treatment_mean - control_mean

print("Treatment mean:", treatment_mean)
print("Control mean:", control_mean)
print("ITT effect:", ITT)

Treatment mean: 17.832711547555075
Control mean: 17.58380804940653
ITT effect: 0.24890349814854318


## 2. Intent-to-Treat (ITT) Estimator

Next, we estimate the **Intent-to-Treat (ITT)** effect.

For the ITT estimator, we compare **everyone who was assigned to the treatment group** with **everyone who was assigned to the control group**, regardless of whether treatment-assigned users actually linked Venmo.

The formula is:

$$
ITT = E[Y \mid T=1] - E[Y \mid T=0]
$$

From the data:

$$
E[Y \mid T=1] = 17.833
$$

and:

$$
E[Y \mid T=0] = 17.584
$$

Therefore:

$$
ITT = 17.833 - 17.584
$$

which gives:

$$
\boxed{ITT \approx 0.249}
$$

So, being assigned the option to link Venmo increased average revenue by approximately:

$$
\$0.25 \text{ per user}
$$

### Interpretation of ITT

The ITT measures the effect of **being assigned or offered access to the Venmo-linking feature**.

It does **not** directly measure the effect of actually linking Venmo.

This distinction is important because many users assigned to the treatment group did not comply.

Therefore, the treatment group includes both:

* Users who were assigned treatment and actually linked Venmo.
* Users who were assigned treatment but did not link Venmo.

So:

$$
\boxed{\text{ITT = Effect of being offered/accessing the treatment option}}
$$

rather than:

$$
\text{Effect of actually using the treatment}
$$

### Why ITT Is Important

The main advantage of the ITT estimator is that it preserves the original **randomization** of the experiment.

Treatment assignment was randomized:

$$
T=1 \quad \text{vs.} \quad T=0
$$

Therefore, the treatment and control groups should be comparable on average before the experiment.

Unlike the As-Treated and Per-Protocol estimators, ITT does not regroup users based on their compliance behavior.

Because of this, ITT generally provides a clean causal estimate of the effect of **offering the Venmo feature**.

However, when compliance is very low, the ITT effect can be much smaller than the effect of actually receiving the treatment because most treatment-assigned users never actually use the feature.


In [6]:
# Effect of assignment on revenue
ITT = (
    df[df["treatment"] == 1]["revenue"].mean()
    - df[df["treatment"] == 0]["revenue"].mean()
)

# Effect of assignment on actually linking Venmo
compliance_effect = (
    df[df["treatment"] == 1]["compliance"].mean()
    - df[df["treatment"] == 0]["compliance"].mean()
)

# Wald estimator
wald = ITT / compliance_effect

print("ITT:", ITT)
print("Compliance effect:", compliance_effect)
print("Wald estimator:", wald)

ITT: 0.24890349814854318
Compliance effect: 0.0428
Wald estimator: 5.815502293190262


## 3. Wald Estimator

Next, we estimate the **Wald estimator**. The Wald estimator estimates the causal effect of actually receiving the treatment when not everyone assigned to treatment complies.

The Wald estimator adjusts the Intent-to-Treat effect by the amount that treatment assignment actually increased compliance.

In other words, it scales the ITT effect by the difference in Venmo-linking rates between the treatment and control groups.

The formula is:

$$
\text{Wald}
=
\frac{E[Y \mid T=1]-E[Y \mid T=0]}
{E[C \mid T=1]-E[C \mid T=0]}
$$

The numerator is the **Intent-to-Treat effect on revenue**:

$$
E[Y \mid T=1]-E[Y \mid T=0]
=
17.8327-17.5838
$$

Therefore:

$$
ITT = 0.2489
$$

The denominator is the effect of treatment assignment on compliance.

From the data:

$$
E[C \mid T=1] = 0.0428
$$

and:

$$
E[C \mid T=0] = 0
$$

So the difference in compliance rates is:

$$
0.0428 - 0 = 0.0428
$$

Therefore:

$$
\text{Wald}
=
\frac{0.2489}{0.0428}
$$

which gives:

$$
\boxed{\text{Wald} \approx 5.82}
$$

So the estimated treatment effect is approximately:

$$
\boxed{\$5.82}
$$

in additional revenue.

### Interpretation

The Wald estimator does **not** estimate the average treatment effect for every user.

Instead, under the standard instrumental-variable assumptions, it estimates the treatment effect for **compliers**.

Compliers are users whose decision to link Venmo was changed by being assigned access to the Venmo feature.

Therefore:

$$
\boxed{
\text{Wald Estimate}
=
\text{Estimated effect of actually linking Venmo for compliers}
}
$$

In this experiment, the estimate suggests that actually linking Venmo increases revenue by approximately **$5.82 for users who link Venmo because they were given access to the feature**.

This is also known as the **Local Average Treatment Effect (LATE)** and LATE gives rise to bias and variance trade off.
Because we reduce the number observations to decrease bias and calcute ATE more precisely but on the other hand since we decreasing smaple size, we will increase variance. 


In [7]:
treat = df[df["treatment"] == 1]
control = df[df["treatment"] == 0]

B = 10000
wald_boot = []

np.random.seed(12345)

for i in range(B):

    # Resampling each group with replacement
    treat_b = treat.sample(
        n=len(treat),
        replace=True
    )

    control_b = control.sample(
        n=len(control),
        replace=True
    )

    # ITT effect on revenue
    numerator = (
        treat_b["revenue"].mean()
        - control_b["revenue"].mean()
    )

    # Effect of assignment on compliance
    denominator = (
        treat_b["compliance"].mean()
        - control_b["compliance"].mean()
    )

    # Wald estimate
    wald_b = numerator / denominator

    wald_boot.append(wald_b)


# Bootstraping standard deviation
wald_sd = np.std(wald_boot, ddof=1)

print("Bootstrap SD:", wald_sd)

Bootstrap SD: 2.7385586951874994


## 4. Bootstrap Standard Deviation of the Wald Estimator

In this step, we estimate the **standard deviation of the Wald estimator** using the bootstrap with 10000 iterations.

Because the Wald estimator is a ratio, we bootstrap the **entire Wald calculation** rather than bootstrapping only the numerator or denominator separately.

The Wald estimator is:

$$
\widehat{\text{Wald}}
=
\frac{
\bar{Y}_{T=1}-\bar{Y}_{T=0}
}{
\bar{C}_{T=1}-\bar{C}_{T=0}
}
$$

where:

* $\bar{Y}_{T=1}$ is the average revenue in the treatment group.
* $\bar{Y}_{T=0}$ is the average revenue in the control group.
* $\bar{C}_{T=1}$ is the compliance rate in the treatment group.
* $\bar{C}_{T=0}$ is the compliance rate in the control group.

The original Wald estimate was:

$$
\boxed{\widehat{\text{Wald}} \approx 5.82}
$$

### Bootstrap Procedure

We use **10,000 bootstrap iterations**.

For each bootstrap iteration, we:

1. Resample the 5,000 treatment users **with replacement**.
2. Resample the 5,000 control users **with replacement**.
3. Calculate the difference in average revenue between the resampled treatment and control groups.
4. Calculate the difference in compliance rates between the resampled treatment and control groups.
5. Divide the revenue difference by the compliance difference to obtain a new Wald estimate.

We resample the treatment and control groups separately so that each bootstrap sample preserves the original experimental structure:

$$
n_{T=1}=5000,
\qquad
n_{T=0}=5000
$$

After 10,000 iterations, we obtain:

$$
\widehat{\text{Wald}}_1,
\widehat{\text{Wald}}_2,
\ldots,
\widehat{\text{Wald}}_{10000}
$$

We then calculate the standard deviation of these 10,000 bootstrap estimates:

$$
SD
\left(
\widehat{\text{Wald}}_1,
\widehat{\text{Wald}}_2,
\ldots,
\widehat{\text{Wald}}_{10000}
\right)
\approx 2.72
$$

Therefore:

$$
\boxed{SE_{\text{bootstrap}}(\widehat{\text{Wald}}) \approx 2.72}
$$

### Interpretation

Our estimated Wald treatment effect is approximately:

$$
\boxed{\$5.82}
$$

with a bootstrap standard error of approximately:

$$
\boxed{\$2.72}
$$

This means that although the estimated treatment effect for compliers is about **$5.82 in additional revenue**, the estimate has substantial sampling variability.

In repeated samples from the same underlying population, the Wald estimate could vary considerably around $5.82.

One important reason for this relatively large standard error is the **very low compliance rate of approximately 4.28%**. Since the Wald estimator divides the ITT effect by the compliance-rate difference,

$$
\widehat{\text{Wald}}
=
\frac{ITT}{0.0428},
$$

a small denominator amplifies sampling variation in the numerator and denominator, making the Wald estimate relatively noisy.
 the bootstrap standard deviation is approximately $2.72.


In [9]:
# Percentiles we want
percentiles = [0.25, 0.50, 0.75, 0.95]

# Signal threshold for each percentile
thresholds = df["signal"].quantile(percentiles)

print("Signal thresholds:")
print(thresholds)

Signal thresholds:
0.25    3.378082
0.50    4.055448
0.75    4.732903
0.95    5.728499
Name: signal, dtype: float64


In [10]:
np.random.seed(12345)

B = 5000      #  5000 bootstrap ite

results = []

for p, threshold in thresholds.items():

    
    # Keeping users above signal threshold
    sub = df[df["signal"] >= threshold]

    # Separating treatment and control
    treat = sub[sub["treatment"] == 1]
    control = sub[sub["treatment"] == 0]

    # Original Wald estimator

    numerator = (
        treat["revenue"].mean()
        - control["revenue"].mean()
    )

    denominator = (
        treat["compliance"].mean()
        - control["compliance"].mean()
    )

    wald = numerator / denominator

    # 3. Bootstraping Wald estimator

    boot_wald = []

    for b in range(B):

        # Resampling treatment users
        treat_b = treat.sample(
            n=len(treat),
            replace=True
        )

        # Resampling control users
        control_b = control.sample(
            n=len(control),
            replace=True
        )

        # Revenue difference
        num_b = (
            treat_b["revenue"].mean()
            - control_b["revenue"].mean()
        )

        # Compliance difference
        den_b = (
            treat_b["compliance"].mean()
            - control_b["compliance"].mean()
        )

        # Wald estimate
        wald_b = num_b / den_b

        boot_wald.append(wald_b)

    # Bootstraping standard error
    boot_se = np.std(boot_wald, ddof=1)

    # result
    results.append({
        "percentile": p,
        "signal_threshold": threshold,
        "sample_size": len(sub),
        "treatment_n": len(treat),
        "control_n": len(control),
        "treatment_compliance": treat["compliance"].mean(),
        "wald": wald,
        "bootstrap_se": boot_se
    })


results = pd.DataFrame(results)

print(results)

   percentile  signal_threshold  sample_size  treatment_n  control_n  \
0        0.25          3.378082         7500         3773       3727   
1        0.50          4.055448         5000         2497       2503   
2        0.75          4.732903         2500         1266       1234   
3        0.95          5.728499          500          249        251   

   treatment_compliance      wald  bootstrap_se  
0              0.054333  4.425704      2.515721  
1              0.072087  3.905233      2.269701  
2              0.103476  2.426411      2.308448  
3              0.200803  2.934658      2.669176  


## 5. Improving Precision by Filtering on the Compliance Signal

In this step, we investigate whether the Wald estimator can become more precise by focusing on users who are **more likely to comply** with the treatment.

### What Are We Doing?

We are given a variable called `signal` that predicts compliance.

Users with higher values of `signal` are more likely to actually link Venmo.

Therefore:

$$
\text{Higher Signal}
\rightarrow
\text{Higher Probability of Compliance}
$$

Instead of using all 10,000 users, we progressively restrict the sample to users with higher values of `signal`.

For example, suppose the 25th percentile of `signal` is:

$$
3.4
$$

Then we keep only users satisfying:

$$
\text{signal} \geq 3.4
$$

Because 25% of users are below this threshold, this keeps approximately the **top 75% of users**.

Importantly, we apply the **same signal threshold to both the treatment and control groups**.

This allows us to preserve the comparison between treatment and control within the selected subset.

### Wald Estimator Within Each Subset

For each signal threshold, we calculate the Wald estimator:

$$
\widehat{\text{Wald}}
=
\frac{
\bar{Y}_{T=1}-\bar{Y}_{T=0}
}{
\bar{C}_{T=1}-\bar{C}_{T=0}
}
$$

We then bootstrap the Wald estimator within that subset to estimate its standard error.

### Why Might Filtering Help?

The idea is that selecting users with higher signal values should increase the treatment-group compliance rate.

Therefore:

$$
\text{Higher Signal}
\rightarrow
\text{Higher Compliance}
\rightarrow
\text{Larger Wald Denominator}
\rightarrow
\text{Possibly Smaller Standard Error}
$$

A larger compliance difference can make the Wald estimator more stable because we are dividing by a larger denominator.

### The Trade-Off

However, filtering also reduces the sample size.

As we increase the signal threshold, we keep fewer users.

For example:

$$
10{,}000
\rightarrow
7{,}500
\rightarrow
5{,}000
\rightarrow
2{,}500
\rightarrow
500
$$

approximately corresponding to increasingly strict signal thresholds.

Therefore, there are two competing effects:

* **Higher compliance improves precision.**
* **Smaller sample size reduces precision.**

The goal is to determine which effect dominates at each threshold.

### What Should We Examine?

For each signal threshold, the main quantities of interest are:

$$
\boxed{\text{Signal Threshold}}
$$

$$
\boxed{\text{Compliance Rate}}
$$

and

$$
\boxed{\text{Bootstrap Standard Error}}
$$

For example, suppose we move from the 25th percentile threshold to the 50th percentile threshold.

If the compliance rate increases and the bootstrap standard error decreases, then filtering has improved precision.

However, at a very high threshold such as the 95th percentile, the compliance rate may be much higher while the remaining sample contains only about 500 users.

At that point, the loss in sample size may become so large that the standard error starts increasing again.

Therefore, the best threshold is not necessarily the one with the highest compliance rate.

Instead, the best threshold is the one that achieves the best balance between:

$$
\boxed{\text{Higher Compliance}}
\qquad \text{and} \qquad
\boxed{\text{Sufficient Sample Size}}
$$

and produces the **smallest bootstrap standard error of the Wald estimator**.


In [11]:
percentiles = [0.25, 0.50, 0.75, 0.95]

for q in percentiles:

    # signal threshold
    threshold = df["signal"].quantile(q)

    # keep users above threshold
    sub = df[df["signal"] >= threshold]

    # treatment group only
    treat = sub[sub["treatment"] == 1]

    # n = treatment group size
    n = len(treat)

    # p = compliance proportion
    p = treat["compliance"].mean()

    # np^2
    np2 = n * (p ** 2)

    print("Percentile:", q)
    print("n:", n)
    print("Compliance p:", p)
    print("np^2:", np2)
    print()

Percentile: 0.25
n: 3773
Compliance p: 0.054333421680360455
np^2: 11.138351444473892

Percentile: 0.5
n: 2497
Compliance p: 0.07208650380456548
np^2: 12.975570684821786

Percentile: 0.75
n: 1266
Compliance p: 0.10347551342812006
np^2: 13.555292259083727

Percentile: 0.95
n: 249
Compliance p: 0.20080321285140562
np^2: 10.040160642570282



# 6. Compliance Proportion and \(np^2\) for Each Signal Threshold

In this step, we examine each signal threshold and its corresponding subset of users. For each subset, we report:

1. The **compliance proportion** in the treatment group.
2. The value of **\(np^2\)** in the treatment group.

These quantities are useful because the approximate variance of the Wald estimator is:

$$
V\!\left(\hat{\beta}\right)=\frac{2}{np^2}\sigma^2
$$

where:

* \(n\) is the number of treated observations,
* \(p\) is the compliance proportion,
* \(\sigma^2\) is the outcome variance.

This formula shows that, holding \(\sigma^2\) fixed, the variance of the Wald estimator decreases when \(np^2\) becomes larger.

Therefore:

$$
\boxed{\text{Larger } np^2 \;\Rightarrow\; \text{smaller variance and smaller standard error}}
$$

### Compliance Proportion

For each subset, the compliance proportion in the treatment group is calculated as:

$$
p=
\frac{\text{number of treatment users who linked Venmo}}
{\text{number of treatment users in that subset}}
$$

### Results

The following results were obtained for the four signal thresholds:

| Signal Percentile Threshold | \(n\) (treated users in subset) | Compliance Proportion \(p\) | \(np^2\) |
| --------------------------- | ------------------------------: | --------------------------: | -------: |
| 25th percentile             |                            3773 |                      0.0543 |  11.1384 |
| 50th percentile             |                            2497 |                      0.0721 |  12.9756 |
| 75th percentile             |                            1266 |                      0.1035 |  13.5553 |
| 95th percentile             |                             249 |                      0.2008 |  10.0402 |

### Interpretation

These results show a clear pattern:

* As the signal threshold increases, the **compliance proportion \(p\)** also increases.
* This means higher-signal users are indeed more likely to comply and link Venmo.
* However, as the threshold increases, the number of treated users \(n\) becomes much smaller.

This creates a trade-off:

* Higher \(p\) helps reduce the variance.
* Smaller \(n\) increases the variance.

The combined effect is summarized by \(np^2\).

From the table, we see that:

* \(np^2\) increases from the 25th percentile to the 75th percentile.
* \(np^2\) reaches its largest value at the **75th percentile**:

$$
np^2 = 13.5553
$$

* At the 95th percentile, even though compliance is much higher, the sample size becomes too small, and \(np^2\) falls to:

$$
np^2 = 10.0402
$$

### Conclusion

Based on the variance approximation

$$
V\!\left(\hat{\beta}\right)=\frac{2}{np^2}\sigma^2,
$$

the **75th percentile threshold** should give the **smallest variance** and therefore the **smallest standard error** among these four options, because it has the largest value of \(np^2\).

In short:

* Filtering on signal improves compliance.
* But filtering too aggressively removes too many users.
* Among the tested thresholds, the **75th percentile appears to provide the best balance between compliance and sample size**.
